# Task 6: End-to-End ML Pipeline

**scikit-learn Pipeline | No Data Leakage | Hyperparameter Tuning | Save Model**

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, roc_auc_score,
                             f1_score, confusion_matrix)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

print("All imports successful")

All imports successful


## Step 1: Data Ingestion

In [4]:
# ── Data Ingestion ─────────────────────────────────────────────
def ingest_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df.drop("customerID", axis=1, inplace=True)
    print(f"  Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"  Missing: {df.isnull().sum().sum()} total")
    return df

df = ingest_data("../data/telco_churn.csv")
df.head(2)

  Loaded: 7043 rows × 20 columns
  Missing: 11 total


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No


## Step 2: EDA Summary

In [5]:
print(f"Churn rate: {(df['Churn']=='Yes').mean()*100:.1f}%  (imbalanced!)")
print(f"Numeric cols:    {df.select_dtypes('number').columns.tolist()}")
print(f"Categorical cols:{df.select_dtypes('object').columns.drop('Churn').tolist()}")

Churn rate: 26.5%  (imbalanced!)
Numeric cols:    ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical cols:['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## Step 3: Feature Engineering inside Pipeline

In [6]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from src.churn_utils import ChurnFeatureEngineer
print("ChurnFeatureEngineer imported from src/churn_utils.py")
print("This ensures the saved model can be loaded anywhere!")

ChurnFeatureEngineer imported from src/churn_utils.py
This ensures the saved model can be loaded anywhere!


## Step 4: Build sklearn Pipeline

In [7]:
# ── Prepare data ──────────────────────────────────────────────
target_enc = LabelEncoder()
df["Churn"] = target_enc.fit_transform(df["Churn"])
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# ── Define column groups ──────────────────────────────────────
num_cols = ["tenure","MonthlyCharges","TotalCharges",
            "ChargesPerTenure","IsEarlyLifecycle","AboveAvgCharges","ValueRatio"]
cat_cols = X.select_dtypes("object").columns.tolist()

# ── Preprocessor ──────────────────────────────────────────────
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# Note: ColumnTransformer runs AFTER ChurnFeatureEngineer adds new columns
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols),
], remainder="drop")

# ── Full Pipeline (Feature Eng → Preprocess → SMOTE → Model) ──
pipe = ImbPipeline([
    ("feature_eng",  ChurnFeatureEngineer()),
    ("preprocessor", preprocessor),
    ("smote",        SMOTE(random_state=42)),
    ("model",        XGBClassifier(eval_metric="logloss", use_label_encoder=False, random_state=42)),
])

print("Pipeline created")
print(pipe)

Pipeline created
Pipeline(steps=[('feature_eng', ChurnFeatureEngineer()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges',
                                                   'ChargesPerTenure',
                                                   'IsEarlyLifecycle',
                                                   'AboveAvgCharges',
                                                   'ValueRatio']),
                                                 ('cat',
         

## Step 5: Hyperparameter Tuning

In [9]:
param_dist = {
    "model__n_estimators":   [100, 200, 300],
    "model__learning_rate":  [0.01, 0.05, 0.1],
    "model__max_depth":      [3, 5, 7],
    "model__subsample":      [0.7, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.9, 1.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    pipe, param_dist, n_iter=15, cv=cv, scoring="roc_auc",
    random_state=42, n_jobs=-1, verbose=1
)
search.fit(X_train, y_train)
print(f"\nBest AUC-ROC (CV): {search.best_score_:.4f}")
print(f"Best params: {search.best_params_}")

Fitting 5 folds for each of 15 candidates, totalling 75 fits

Best AUC-ROC (CV): 0.8474
Best params: {'model__subsample': 0.7, 'model__n_estimators': 300, 'model__max_depth': 3, 'model__learning_rate': 0.01, 'model__colsample_bytree': 0.7}


## Step 6: Evaluation

In [10]:
best_pipe = search.best_estimator_
y_pred = best_pipe.predict(X_test)
y_prob = best_pipe.predict_proba(X_test)[:, 1]

print("FINAL MODEL EVALUATION (Held-out Test Set)")

print(classification_report(y_test, y_pred, target_names=["No Churn","Churn"]))
print(f"AUC-ROC:  {roc_auc_score(y_test, y_prob):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")

FINAL MODEL EVALUATION (Held-out Test Set)
              precision    recall  f1-score   support

    No Churn       0.90      0.76      0.82      1035
       Churn       0.53      0.76      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.76      0.77      1409

AUC-ROC:  0.8409
F1-Score: 0.6268


## Step 7: Save Model

In [ ]:
joblib.dump(best_pipe, "../models/churn_pipeline.pkl")
print("Full pipeline saved to models/churn_pipeline.pkl")

# Test reload
reloaded = joblib.load("../models/churn_pipeline.pkl")
test_sample = X_test.iloc[:3]
pred = reloaded.predict(test_sample)
prob = reloaded.predict_proba(test_sample)[:,1]
print("\nReload test — predictions:", pred, "| probabilities:", prob.round(3))
print("\nPipeline is deployment-ready!")

Full pipeline saved to models/churn_pipeline.pkl
Note: ChurnFeatureEngineer is in src/churn_utils.py — imported before joblib.load() anywhere

Reload test — predictions: [0 1 0] | probabilities: [0.111 0.72  0.134]

Pipeline is deployment-ready!
